In [0]:
tables = {
    "drivers": spark.table("workspace.logistics_project.drivers"),
    "trucks": spark.table("workspace.logistics_project.trucks"),
    "trailers": spark.table("workspace.logistics_project.trailers"),
    "customers": spark.table("workspace.logistics_project.customers"),
    "facilities": spark.table("workspace.logistics_project.facilities"),
    "routes": spark.table("workspace.logistics_project.routes"),
    "loads": spark.table("workspace.logistics_project.loads"),
    "trips": spark.table("workspace.logistics_project.trips"),
    "fuel_purchases": spark.table("workspace.logistics_project.fuel_purchases"),
    "maintenance_records": spark.table("workspace.logistics_project.maintenance_records"),
    "delivery_events": spark.table("workspace.logistics_project.delivery_events"),
    "safety_incidents": spark.table("workspace.logistics_project.safety_incidents"),
    "driver_monthly_metrics": spark.table("workspace.logistics_project.driver_monthly_metrics"),
    "truck_utilization_metrics": spark.table("workspace.logistics_project.truck_utilization_metrics")
}

In [0]:
#Read Tables
drivers= spark.table("workspace.logistics_project.drivers")
trucks= spark.table("workspace.logistics_project.trucks")
trailers= spark.table("workspace.logistics_project.trailers")
customers= spark.table("workspace.logistics_project.customers")
facilities= spark.table("workspace.logistics_project.facilities")
routes= spark.table("workspace.logistics_project.routes")
loads= spark.table("workspace.logistics_project.loads")
trips= spark.table("workspace.logistics_project.trips")
fuel_purchases= spark.table("workspace.logistics_project.fuel_purchases")
maintenance_records= spark.table("workspace.logistics_project.maintenance_records")
delivery_events= spark.table("workspace.logistics_project.delivery_events")
safety_incidents= spark.table("workspace.logistics_project.safety_incidents")
driver_monthly_metrics= spark.table("workspace.logistics_project.driver_monthly_metrics")
truck_utilization_metrics= spark.table("workspace.logistics_project.truck_utilization_metrics")

In [0]:
#Check Row Counts
for name, df in tables.items():
    print(f"{name}: {df.count()} rows")

drivers: 150 rows
trucks: 120 rows
trailers: 180 rows
customers: 200 rows
facilities: 50 rows
routes: 58 rows
loads: 85410 rows
trips: 85410 rows
fuel_purchases: 374222 rows
maintenance_records: 2920 rows
delivery_events: 170820 rows
safety_incidents: 170 rows
driver_monthly_metrics: 4464 rows
truck_utilization_metrics: 3312 rows


In [0]:
#Check Null Values
from pyspark.sql.functions import *

for name, df in tables.items():
    print(f"\n{name.upper()}")

    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()


DRIVERS
+---------+----------+---------+---------+----------------+--------------+-------------+-------------+-------------+-----------------+---------+----------------+
|driver_id|first_name|last_name|hire_date|termination_date|license_number|license_state|date_of_birth|home_terminal|employment_status|cdl_class|years_experience|
+---------+----------+---------+---------+----------------+--------------+-------------+-------------+-------------+-----------------+---------+----------------+
|        0|         0|        0|        0|             124|             0|            0|            0|            0|                0|        0|               0|
+---------+----------+---------+---------+----------------+--------------+-------------+-------------+-------------+-----------------+---------+----------------+


TRUCKS
+--------+-----------+----+----------+---+----------------+-------------------+---------+---------------------+------+-------------+
|truck_id|unit_number|make|model_year|v

In [0]:
#Remove Duplicates
cleaned_tables = {}

for name, df in tables.items():
    cleaned_tables[name] = df.dropDuplicates()

In [0]:
#Compare Counts
for name in tables.keys():

    original = tables[name].count()
    cleaned = cleaned_tables[name].count()

    print(f"{name}")
    print(f"Original: {original}")
    print(f"Cleaned : {cleaned}")
    print("-"*40)

drivers
Original: 150
Cleaned : 150
----------------------------------------
trucks
Original: 120
Cleaned : 120
----------------------------------------
trailers
Original: 180
Cleaned : 180
----------------------------------------
customers
Original: 200
Cleaned : 200
----------------------------------------
facilities
Original: 50
Cleaned : 50
----------------------------------------
routes
Original: 58
Cleaned : 58
----------------------------------------
loads
Original: 85410
Cleaned : 85410
----------------------------------------
trips
Original: 85410
Cleaned : 85410
----------------------------------------
fuel_purchases
Original: 374222
Cleaned : 374222
----------------------------------------
maintenance_records
Original: 2920
Cleaned : 2920
----------------------------------------
delivery_events
Original: 170820
Cleaned : 170820
----------------------------------------
safety_incidents
Original: 170
Cleaned : 170
----------------------------------------
driver_monthly_metrics

In [0]:
#Create Silver Schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.logistics_silver
""")

DataFrame[]

In [0]:
spark.sql("""
SHOW TABLES IN workspace.logistics_silver
""").show(truncate=False)

+----------------+---------------+-----------+
|database        |tableName      |isTemporary|
+----------------+---------------+-----------+
|logistics_silver|delivery_events|false      |
|logistics_silver|drivers        |false      |
|logistics_silver|loads          |false      |
|logistics_silver|trips          |false      |
+----------------+---------------+-----------+



In [0]:
spark.table("workspace.logistics_project.routes").printSchema()

spark.table("workspace.logistics_project.customers").printSchema()

spark.table("workspace.logistics_project.maintenance_records").printSchema()

spark.table("workspace.logistics_project.safety_incidents").printSchema()

spark.table("workspace.logistics_project.fuel_purchases").printSchema()

root
 |-- route_id: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- typical_distance_miles: long (nullable = true)
 |-- base_rate_per_mile: double (nullable = true)
 |-- fuel_surcharge_rate: double (nullable = true)
 |-- typical_transit_days: long (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- credit_terms_days: long (nullable = true)
 |-- primary_freight_type: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- contract_start_date: date (nullable = true)
 |-- annual_revenue_potential: long (nullable = true)

root
 |-- maintenance_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- maintenance_date: date (nullable = true)
 |-- maintenance_type: string (nullable = t